In [54]:
from datetime import date, datetime, timedelta
from dateutil.relativedelta import relativedelta
from typing import TypedDict, List, Optional
from langgraph.graph import StateGraph, START, END
from langgraph.types import Send, interrupt, Command

from openai import OpenAI
from langchain.chat_models import init_chat_model
from langchain_core.messages import SystemMessage, HumanMessage
from pydantic import BaseModel, Field
from typing_extensions import Annotated
import operator
import math
import requests
import json

# from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.checkpoint.memory import InMemorySaver
memory = InMemorySaver()

from prompt import prompt, SUMMARY_SYSTEM_PROMPT

llm = init_chat_model("openai:gpt-4o-mini")

# ---- State ----
class State(TypedDict):
    raw_query: str
    topic: str
    start_date: str
    end_date: str
    count: int
    papers: Annotated[list[dict], operator.add]
    summary: Annotated[list[dict], operator.add]
    selected_id: int
    # assumption_qna: List[dict]
    # graph_rag_output: dict

In [55]:
class ExtractedQueryParams(BaseModel):
    topic: str = Field(description="검색할 논문 주제/키워드")
    year_delta: int = Field(description="검색 기간 년")
    month_delta: int = Field(description="검색 기간 월")
    day_delta: int = Field(description="검색 기간 일")
    count: int = Field(description="수집할 논문 개수")

def llm_extract_params(raw_query: str) -> dict:
    structured_llm = llm.with_structured_output(ExtractedQueryParams)
    result = structured_llm.invoke( prompt + raw_query)
    return result.model_dump()

# ---- 1. 파라미터 파싱 ----
def parse_query(state: State):
    parsed = llm_extract_params(state["raw_query"])
    today = date.today()

    start_date = today - relativedelta(
    years=parsed["year_delta"],
    months=parsed["month_delta"],
    days=parsed["day_delta"],
    )
    return {
        "topic": parsed["topic"],
        "start_date": start_date.isoformat(),
        "end_date": today.isoformat(),
        "count": parsed["count"],
    }

In [ ]:
today = date.today()
parsed = {'topic': 'Agent', 'year_delta': 0, 'month_delta': 8, 'day_delta': 0, 'count': 5}
start_date = (today - relativedelta(
    years=int(parsed["year_delta"]),
    months=int(parsed["month_delta"]),
    days=int(parsed["day_delta"]),
    )).isoformat()
print (start_date)

2025-11-07


In [56]:
HF_DAILY_PAPERS_URL = "https://huggingface.co/api/daily_papers"
DAILY_PAPERS_PAGE_LIMIT = 100  # daily_papers 응답 페이지당 최대 개수
MIN_UPVOTES = 110  # 이 값 미만인 논문은 후보에서 제외
SCORE_WEIGHT_VOTES = 0.45
SCORE_WEIGHT_GITHUB_STARS = 0.35
SCORE_WEIGHT_RECENCY = 0.2
MIN_CANDIDATES_NUM = 50

In [57]:
def fetch_from_huggingface(topic: str, start_date: str, end_date: str, count: int) -> list[dict]:
    topic_lower = topic.lower()
    first = date.fromisoformat(start_date)
    current = date.fromisoformat(end_date)

    candidates = []
    max_num = max(MIN_CANDIDATES_NUM, count*10)
    while current > first:
        response = requests.get(
            HF_DAILY_PAPERS_URL,
            params={"date": current.isoformat(), "limit": DAILY_PAPERS_PAGE_LIMIT},
            timeout=10,
        )
        try:
            response.raise_for_status()
        except requests.HTTPError:
            if response.status_code == 400:
                # HF가 아직 발행하지 않은/허용하지 않는 날짜 (예: 오늘) -> 이 날짜만 건너뜀
                current -= timedelta(days=1)
                continue
            raise
        for item in response.json():
            paper = item["paper"]
            upvotes = paper.get("upvotes") or 0
            if upvotes < MIN_UPVOTES:
                break
            haystack = (paper["title"] + paper["summary"]).lower()
            if topic_lower not in haystack:
                continue
            candidates.append({
                "title": paper["title"],
                "summary": paper["summary"],
                "upvotes": upvotes,
                "github_stars": paper.get("githubStars") or 0,
                "github_repo": paper.get("githubRepo"),
                "published_at": paper["publishedAt"],
            })
        current -= timedelta(days=1)
        if len(candidates) > max_num:
            break

    return candidates

In [58]:
def score_and_sort(papers: list[dict]) -> list[dict]:
    if not papers:
        return []

    def normalize(values: list[float]) -> list[float]:
        low, high = min(values), max(values)
        if high == low:
            return [1.0 for _ in values]
        return [(v - low) / (high - low) for v in values]

    vote_scores = normalize([math.log1p(p["upvotes"]) for p in papers])
    star_scores = normalize([math.log1p(p["github_stars"]) for p in papers])
    published_ts = [
        datetime.fromisoformat(p["published_at"]).timestamp() for p in papers
    ]
    recency_scores = normalize(published_ts)

    scored = []
    for paper, vote_score, star_score, recency_score in zip(
        papers, vote_scores, star_scores, recency_scores
    ):
        score = (
            SCORE_WEIGHT_VOTES * vote_score
            + SCORE_WEIGHT_GITHUB_STARS * star_score
            + SCORE_WEIGHT_RECENCY * recency_score
        )
        scored.append({**paper, "score": score})

    scored.sort(key=lambda p: p["score"], reverse=True)
    return scored

# ---- 2-2. 수집 + 스코어링 노드 ----
def fetch_and_score(state: State):
    raw_papers = fetch_from_huggingface(
        state["topic"], state["start_date"], state["end_date"], state["count"]
    )
    raw_papers = score_and_sort(raw_papers)
    if len(raw_papers) > state["count"]:
        scored = raw_papers[: state["count"]]
    else:
        scored = raw_papers
    return {"papers": scored}

In [59]:
def dispatch_summarizers(state: State):
    papers = state["papers"]
    meta_infos = []
    for i, paper in enumerate(papers):
        meta_infos.append({"id": i + 1, "paper": paper})
    return [Send("summarize_paper", meta_info) for meta_info in meta_infos]

class PaperReview(BaseModel):
    rating: float = Field(description="1~5점")

    why_read: str
    why_skip: str

    key_takeaways: list[str]

    application_ideas: list[str]

    summmary: str

SYSTEM_MESSAGE = SystemMessage(content=SUMMARY_SYSTEM_PROMPT)
summary_structured_llm = llm.with_structured_output(PaperReview)
def summarize_paper(meta_info):
    paper_id = meta_info["id"]
    paper = meta_info["paper"]

    result = summary_structured_llm.invoke([
        SYSTEM_MESSAGE,
        HumanMessage(content=json.dumps(paper, ensure_ascii=False))])
    
    parsed = result.model_dump()
    parsed["id"] = paper_id
    parsed["title"] = paper["title"]

    return {
        "summaries": [parsed],
    }

In [66]:
def user_select_paper(state: State):
    answer = interrupt({
        "summary": state["summary"],
        "choice": "Which paper do you like to check?"
        }
    )
    choice = answer["choice"]
    return {"selected_id": int(choice) - 1}

In [61]:
builder = StateGraph(State)
builder.add_node("parse_query", parse_query)
builder.add_node("fetch_and_score", fetch_and_score)
builder.add_node("summarize_paper", summarize_paper)
builder.add_node("user_select_paper", user_select_paper)
# builder.add_node("assumption_checker_agent", assumption_checker_agent)
# builder.add_node("graph_rag_agent", graph_rag_agent)

builder.add_edge(START, "parse_query")
builder.add_edge("parse_query", "fetch_and_score")
builder.add_conditional_edges(
    "fetch_and_score", dispatch_summarizers, ["summarize_paper"]
)
builder.add_edge("summarize_paper", "user_select_paper")
# builder.add_edge("user_select_paper", "assumption_checker_agent")

# builder.add_conditional_edges(
#     "assumption_checker_agent",
#     should_continue_checking,
#     {
#         "assumption_checker_agent": "assumption_checker_agent",
#         "graph_rag_agent": "graph_rag_agent",
#     },
# )

builder.add_edge("user_select_paper", END)

# ---- SqliteSaver 연결 ----
# conn = sqlite3.connect("checkpoints.sqlite", check_same_thread=False)
# checkpointer = SqliteSaver(conn)

# graph = builder.compile(checkpointer=checkpointer)
graph = builder.compile(checkpointer=memory)

In [62]:
config = {
    "configurable": {
        "thread_id": "1",
    },
}

In [64]:
graph.invoke(
    {"raw_query": "최근 2주간 LLM 논문 2개"},
    config=config,
)

{'raw_query': '최근 2주간 LLM 논문 2개',
 'topic': 'LLM',
 'start_date': '2026-06-24',
 'end_date': '2026-07-08',
 'count': 2,
 'papers': [{'title': 'Agentic Abstention: Do Agents Know When to Stop Instead of Act?',
   'summary': "LLM agents are expected to act over multiple turns, using search, browsing interfaces, and terminal tools to complete user goals. Yet not every goal is well specified or achievable in the available environment. In such cases, a reliable agent should recognize that further interaction is unlikely to help and abstain from additional tool calls. We define Agentic Abstention, the problem of deciding when an agent should stop acting under uncertainty. Unlike standard LLM abstention, which is usually evaluated as a single-turn answer-or-abstain decision, agentic abstention is a sequential decision problem: an agent can answer, abstain, or gather more information at each turn, and the need to abstain may only become clear after interacting with the environment. We study th

In [ ]:
response = {
    "choice": 2,
}

graph.invoke(
    Command(resume=response),
    config=config,
)

TypeError: unsupported operand type(s) for -: 'str' and 'int'